# SCoPE — designed-coalition validation + extraction arms (Kaggle)

Builds the chain-split (AND) and hard-traps (OR) cells, validates the designed coalitions
against model behavior, and compares the five extraction arms on query budget vs coverage.

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100) · Internet **on** ·
attach the **scope-repo** dataset (the repo zip) via *Add Input*.

Outputs land in `runs/` and are zipped for download at the end. Rough budget on a T4:
pipeline ~40–60 min per condition, validation ~1 h, extraction ~2–3 h at the default limits —
trim the knobs in cell 2 if the session gets tight.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

# lineup: public clone, pulled on re-run so the notebook stays current.
if (WORK / 'lineup').exists():
    subprocess.run(['git', '-C', 'lineup', 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/santoshcheethiralame-dot/LINEUP', 'lineup'], check=True)

# scope: prefer the attached dataset (repo zip, auto-extracted under /kaggle/input);
# fall back to a git URL in SCOPE_URL if one is set.
scope_src = next((p.parent for p in Path('/kaggle/input').glob('*/**/src/scope/__init__.py')), None)
if scope_src is not None:
    scope_root = scope_src.parent.parent
    shutil.rmtree(WORK / 'scope', ignore_errors=True)
    shutil.copytree(scope_root, WORK / 'scope')
elif os.environ.get('SCOPE_URL'):
    subprocess.run(['git', 'clone', '--depth', '1', os.environ['SCOPE_URL'], 'scope'], check=True)
else:
    raise SystemExit('attach the scope-repo dataset (or set SCOPE_URL)')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './lineup[gpu]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './scope'], check=True)
print('[ok] lineup + scope installed')

In [ ]:
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
TAG = 'qwen'
DATASET = 'hotpotqa'          # hotpotqa | 2wiki | musique
LIMIT = 300                   # source questions per condition (chain-split skips non-bridge cases)
K = 6
SEED = 0
RUN_AND = True                # chain-split condition (the new AND cells)
RUN_OR = True                 # hard-traps condition (redundant OR cover)
VALIDATE_MAX_SIZE = 3         # exact-enumeration bound for validate_designed
VALIDATE_LIMIT = 60           # wrong cases per cell for validation (0 = all)
EXTRACT_LIMIT = 40            # wrong cases per cell for the extraction arms (0 = all)
N_SAMPLES = 48                # surrogate masks for the interaction/beam arms

conditions = ([('and', '--chain-split')] if RUN_AND else []) + ([('hardtraps', '--hard-traps')] if RUN_OR else [])
cells = {name: f'runs/{DATASET}/{name}/{TAG}' for name, _ in conditions}
print(cells)

In [ ]:
# Build + generate + oracle + methods, one condition at a time. Progress streams live.
for name, flag in conditions:
    print(f'==== pipeline: {name} ====', flush=True)
    subprocess.run(
        [sys.executable, 'lineup/scripts/run_pipeline.py',
         '--dataset', DATASET, flag, '--limit', str(LIMIT), '--k', str(K), '--seed', str(SEED),
         '--model', MODEL, '--load-in-4bit', '--out', cells[name]],
        check=True,
    )
    out = Path(cells[name])
    needed = ['scenarios.jsonl', 'generations.jsonl', 'roles.jsonl', 'predictions.jsonl', 'designed.jsonl']
    missing = [f for f in needed if not (out / f).exists()]
    if missing:
        print(f'!!!!!! STOP: {name} missing {missing}')
    else:
        print(f'[ok] {name}: all artifacts written')

In [ ]:
# Does the model behave as the construction designed? Exact minimal sufficient sets vs the plant.
for name, _ in conditions:
    print(f'==== validate_designed: {name} ====', flush=True)
    args = [sys.executable, 'scope/scripts/validate_designed.py', '--cell', cells[name],
            '--model', MODEL, '--load-in-4bit', '--max-size', str(VALIDATE_MAX_SIZE)]
    if VALIDATE_LIMIT:
        args += ['--limit', str(VALIDATE_LIMIT)]
    subprocess.run(args, check=True)

In [ ]:
# The five extraction arms: coverage of the designed set vs queries spent.
for name, _ in conditions:
    print(f'==== run_extraction: {name} ====', flush=True)
    args = [sys.executable, 'scope/scripts/run_extraction.py', '--cell', cells[name],
            '--model', MODEL, '--load-in-4bit', '--n-samples', str(N_SAMPLES), '--seed', str(SEED)]
    if EXTRACT_LIMIT:
        args += ['--limit', str(EXTRACT_LIMIT)]
    subprocess.run(args, check=True)

In [ ]:
# Free check on the AND cells: the designed silent link. If the construction works, the link
# passage should label silent (causal, never salient) and the value passage culprit.
from collections import Counter
from lineup.data.serialization import read_roles, read_scenarios

if RUN_AND:
    scenarios = {s.qid: s for s in read_scenarios(Path(cells['and']) / 'scenarios.jsonl')}
    link_roles, value_roles = Counter(), Counter()
    for case in read_roles(Path(cells['and']) / 'roles.jsonl'):
        if case.original_correct:
            continue
        provenance = {c.chunk_id: c.provenance for c in scenarios[case.qid].chunks}
        for role in case.chunk_roles:
            if provenance.get(role.chunk_id) == 'link':
                link_roles[role.role] += 1
            elif provenance.get(role.chunk_id) == 'misleading':
                value_roles[role.role] += 1
    print('link passage roles :', dict(link_roles))
    print('value passage roles:', dict(value_roles))

In [ ]:
# Zip everything for download, self-labelled.
import shutil
stem = f'scope_{DATASET}_{TAG}_seed{SEED}'
shutil.make_archive(stem, 'zip', 'runs')
print('download:', f'/kaggle/working/{stem}.zip')